Box- or lasso-select on any plot to filter all of them through a single `hv.link_selections` instance.

In [ ]:
import holoviews as hv
import hvplot.pandas  # noqa: F401
import numpy as np
import pandas as pd
import panel as pn

pn.extension()

## Introduction

In [ ]:
art = pn.ui.Column(
    pn.ui.Markdown("## Welcome and meet the Palmer penguins!"),
    pn.ui.PNG('https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/man/figures/palmerpenguins.png', height=160),
    pn.ui.Markdown("### Artwork by @allison_horst"),
    pn.ui.Markdown("""
Use the box-select and lasso-select tools to select a subset of penguins
and reveal more information about the selected subgroup through the power
of cross-filtering.
"""),
    pn.ui.Markdown("""
### License

Data are available by CC-0 license in accordance with the Palmer Station LTER Data Policy and the LTER Data Access Policy for Type I data.
"""),
    sizing_mode='stretch_width'
)

art

## Building some plots

Let us first load the Palmer penguin dataset ([Gorman et al.](https://allisonhorst.github.io/palmerpenguins/)) which contains measurements about a number of penguin species:

In [ ]:
penguins = pd.read_csv('https://datasets.holoviz.org/penguins/v1/penguins.csv')
penguins = penguins[~penguins.sex.isnull()].reset_index().sort_values('species')

penguins.head()

Next we will set up a linked selections instance that will allow us to perform cross-filtering on the plots we will create in the next step. `selection_param` yields the selected rows, so the counter is a bound string:

In [ ]:
ls = hv.link_selections.instance()

def count(selected):
    return f"## {len(selected)}/{len(penguins)} penguins selected"

selected = pn.ui.Typography(
    pn.bind(count, ls.selection_param(penguins)),
    disable_anchors=True,
    sx={"color": "var(--mui-palette-primary-main);"},
    styles={"margin-left": "auto"}
)

selected

Now we can start plotting the data with hvPlot, which provides a familiar API to pandas `.plot` users but generates interactive plots and use the linked selections object to allow cross-filtering across the plots:

In [ ]:
colors = {
    'Adelie': '#1f77b4',
    'Gentoo': '#ff7f0e',
    'Chinstrap': '#2ca02c'
}

scatter = penguins.hvplot.points(
    'bill_length_mm', 'bill_depth_mm', c='species',
    cmap=colors, responsive=True, min_height=300
)

histogram = penguins.hvplot.hist(
    'body_mass_g', by='species', color=hv.dim('species').categorize(colors),
    legend=False, alpha=0.5, responsive=True, min_height=300
)

bars = penguins.hvplot.bar(
    'species', 'index', c='species', cmap=colors,
    responsive=True, min_height=300, ylabel=''
).aggregate(function=np.count_nonzero)

violin = penguins.hvplot.violin(
    'flipper_length_mm', by=['species', 'sex'], cmap='Category20',
    responsive=True, min_height=300, legend='bottom_right'
).opts(split='sex')

plots = pn.ui.HoloViews(
    ls(scatter.opts(show_legend=False) + bars + histogram + violin).opts(sizing_mode='stretch_both').cols(2)
)

## App

In [ ]:
page = pn.ui.Page(
    header=[selected],
    main=[plots],
    sidebar=[art],
    logo='https://github.com/allisonhorst/palmerpenguins/raw/main/man/figures/logo.png',
    title="Palmer Penguins"
)

page

### Served App

In [ ]:
if pn.state.served:
    page.servable()